# Pocket sequence + ProtT5 residue embeddings

Simple workflow:

## Option A — ligand-based pocket
1. Upload a **PDB** or **mmCIF**
2. The notebook detects **crystallographic ligands**
3. It shows a table with candidate pockets
4. You choose a `ligand_index`
5. The pocket sequence is extracted
6. ProtT5 residue embeddings are computed

## Option B — coordinate-based pocket
1. Upload a **PDB** or **mmCIF**
2. Enter a pocket center `(x, y, z)` and radius
3. The pocket sequence is extracted
4. ProtT5 residue embeddings are computed

In [ ]:
# Install dependencies
!pip -q install biopython transformers sentencepiece

In [ ]:
# ====== CONFIG ======
STRUCTURE_FILE = "/content/your_structure.pdb"   # change after uploading your file
POCKET_RADIUS = 10.0
OUTPUT_PREFIX = "selected_pocket"

In [ ]:
# Core utilities
import os
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser, MMCIFParser, is_aa

EXCLUDE_LIG = set("""
HOH WAT DOD NA K CL MG CA MN ZN CU CD CO FE SO4 PO4 NO3
GOL EDO PEG PGE PGO TRS MES HEP HEPES BME DMS ACT ACE CIT
""".split())

THREE_TO_ONE = {
    'ALA':'A','CYS':'C','ASP':'D','GLU':'E','PHE':'F','GLY':'G','HIS':'H',
    'ILE':'I','LYS':'K','LEU':'L','MET':'M','ASN':'N','PRO':'P','GLN':'Q',
    'ARG':'R','SER':'S','THR':'T','VAL':'V','TRP':'W','TYR':'Y'
}

def load_structure(path: str):
    path = os.path.abspath(path)
    name = os.path.basename(path).split(".")[0]
    if path.lower().endswith(".cif") or path.lower().endswith(".mmcif"):
        parser = MMCIFParser(QUIET=True)
    else:
        parser = PDBParser(QUIET=True)
    return parser.get_structure(name, path)

def atom_is_heavy(atom):
    return atom.element not in ("H", "D")

def residue_atoms_heavy(residue):
    return [a for a in residue.get_atoms() if atom_is_heavy(a)]

def residue_id_str(residue):
    hetflag, resseq, icode = residue.get_id()
    icode = icode.strip() if isinstance(icode, str) else ""
    return f"{resseq}{icode}"

def ligand_center(lig_res):
    xyz = [a.get_coord() for a in residue_atoms_heavy(lig_res)]
    return np.mean(xyz, axis=0) if xyz else None

def list_crystal_ligands(structure):
    rows = []
    for model in structure:
        for chain in model:
            for res in chain:
                hetflag, _, _ = res.get_id()
                if hetflag.strip() == "" or is_aa(res, standard=True):
                    continue
                resname = res.get_resname().strip()
                if resname in EXCLUDE_LIG:
                    continue
                heavy_atoms = residue_atoms_heavy(res)
                heavy = len(heavy_atoms)
                if heavy < 6:
                    continue
                center = ligand_center(res)
                rows.append({
                    "ligand_index": len(rows),
                    "model_id": model.get_id(),
                    "chain_id": chain.get_id(),
                    "resname": resname,
                    "residue_id": residue_id_str(res),
                    "heavy_atoms": heavy,
                    "center_x": None if center is None else float(center[0]),
                    "center_y": None if center is None else float(center[1]),
                    "center_z": None if center is None else float(center[2]),
                    "_res_obj": res,
                })
    return rows

def min_distance_residue_to_point(residue, point_xyz):
    atoms = residue_atoms_heavy(residue)
    if not atoms:
        return np.inf
    return min(np.linalg.norm(a.get_coord() - point_xyz) for a in atoms)

def min_distance_residue_to_ligand(residue, ligand_res):
    ra = residue_atoms_heavy(residue)
    la = residue_atoms_heavy(ligand_res)
    if not ra or not la:
        return np.inf
    return min(np.linalg.norm(a.get_coord() - b.get_coord()) for a in ra for b in la)

def extract_pocket_by_ligand(structure, ligand_row, radius=10.0, target_chain=None):
    ligand_res = ligand_row["_res_obj"]
    residues = []
    for model in structure:
        for chain in model:
            if target_chain is not None and chain.get_id() != target_chain:
                continue
            for res in chain:
                if not is_aa(res, standard=True):
                    continue
                d = min_distance_residue_to_ligand(res, ligand_res)
                if d <= radius:
                    residues.append((chain.get_id(), res, float(d)))
    residues.sort(key=lambda x: (x[0], x[1].get_id()[1], x[1].get_id()[2]))
    return residues

def extract_pocket_by_coords(structure, center_xyz, radius=10.0, target_chain=None):
    residues = []
    center_xyz = np.asarray(center_xyz, dtype=float)
    for model in structure:
        for chain in model:
            if target_chain is not None and chain.get_id() != target_chain:
                continue
            for res in chain:
                if not is_aa(res, standard=True):
                    continue
                d = min_distance_residue_to_point(res, center_xyz)
                if d <= radius:
                    residues.append((chain.get_id(), res, float(d)))
    residues.sort(key=lambda x: (x[0], x[1].get_id()[1], x[1].get_id()[2]))
    return residues

def pocket_to_dataframe(pocket_residues):
    rows = []
    for chain_id, res, dist in pocket_residues:
        rows.append({
            "chain_id": chain_id,
            "residue_number": res.get_id()[1],
            "insertion_code": res.get_id()[2].strip() if isinstance(res.get_id()[2], str) else "",
            "resname_3l": res.get_resname().strip(),
            "resname_1l": THREE_TO_ONE.get(res.get_resname().strip(), "X"),
            "distance": dist,
        })
    return pd.DataFrame(rows)

def pocket_sequences(pocket_residues):
    seq3 = [res.get_resname().strip() for _, res, _ in pocket_residues]
    seq1 = "".join(THREE_TO_ONE.get(x, "X") for x in seq3)
    return seq3, seq1

In [ ]:
# Load structure
structure = load_structure(STRUCTURE_FILE)
print("Loaded structure:", STRUCTURE_FILE)

In [ ]:
# Show ligand-based candidate pockets
ligands = list_crystal_ligands(structure)

if len(ligands) == 0:
    print("No crystallographic ligands detected.")
else:
    preview_rows = []
    for i, lig in enumerate(ligands):
        pocket = extract_pocket_by_ligand(structure, lig, radius=POCKET_RADIUS)
        seq3, seq1 = pocket_sequences(pocket)

        preview_rows.append({
            "ligand_index": i,
            "chain": lig["chain_id"],
            "resname": lig["resname"],
            "residue_id": lig["residue_id"],
            "heavy_atoms": lig["heavy_atoms"],
            "pocket_n_residues": len(pocket),
            "pocket_seq_1L": seq1,
            "pocket_seq_3L": " ".join(seq3),
        })

    ligands_df = pd.DataFrame(preview_rows)
    display(ligands_df)

print("Choose a ligand_index and run the next cell, or use coords mode.")

In [ ]:
# ====== USER CHOICE ======
MODE = "ligand"   # "ligand" or "coords"

# If MODE == "ligand"
SELECT_LIGAND_INDEX = 0

# If MODE == "coords"
CENTER = (0.0, 0.0, 0.0)
TARGET_CHAIN = None   # e.g. "A" or None

In [ ]:
# Build pocket from user choice
if MODE == "ligand":
    if len(ligands) == 0:
        raise ValueError("No ligands available. Switch to coords mode.")
    if SELECT_LIGAND_INDEX < 0 or SELECT_LIGAND_INDEX >= len(ligands):
        raise ValueError(f"Invalid SELECT_LIGAND_INDEX: {SELECT_LIGAND_INDEX}")

    selected_lig = ligands[SELECT_LIGAND_INDEX]
    pocket_residues = extract_pocket_by_ligand(
        structure,
        selected_lig,
        radius=POCKET_RADIUS,
        target_chain=TARGET_CHAIN,
    )

    selection_info = {
        "mode": "ligand",
        "selected_ligand_index": SELECT_LIGAND_INDEX,
        "ligand_chain": selected_lig["chain_id"],
        "ligand_resname": selected_lig["resname"],
        "ligand_residue_id": selected_lig["residue_id"],
        "target_chain": TARGET_CHAIN,
    }

elif MODE == "coords":
    pocket_residues = extract_pocket_by_coords(
        structure,
        center_xyz=CENTER,
        radius=POCKET_RADIUS,
        target_chain=TARGET_CHAIN,
    )

    selection_info = {
        "mode": "coords",
        "center_x": CENTER[0],
        "center_y": CENTER[1],
        "center_z": CENTER[2],
        "target_chain": TARGET_CHAIN,
    }

else:
    raise ValueError("MODE must be 'ligand' or 'coords'")

if len(pocket_residues) == 0:
    raise ValueError("No residues found for the selected pocket.")

print("Selected pocket:", selection_info)

In [ ]:
# Pocket sequence and residue table
pocket_df = pocket_to_dataframe(pocket_residues)
seq3, seq1 = pocket_sequences(pocket_residues)

display(pocket_df.head(50))
print("Pocket sequence (3-letter):")
print(" ".join(seq3))
print("\nPocket sequence (1-letter):")
print(seq1)
print("\nNumber of residues:", len(seq1))

pocket_df.to_csv(f"{OUTPUT_PREFIX}_residues.csv", index=False)
with open(f"{OUTPUT_PREFIX}_sequence_1l.txt", "w") as f:
    f.write(seq1 + "\n")
with open(f"{OUTPUT_PREFIX}_sequence_3l.txt", "w") as f:
    f.write(" ".join(seq3) + "\n")

In [ ]:
# Load ProtT5
import torch
from transformers import T5EncoderModel, T5Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_name = "Rostlab/prot_t5_xl_uniref50"
tokenizer = T5Tokenizer.from_pretrained(model_name, do_lower_case=False)
model = T5EncoderModel.from_pretrained(model_name).to(device)
model.eval()

In [ ]:
# Optional debug: inspect ProtT5 tokenization
def inspect_prott5_tokenization(seq_1l: str):
    seq_spaced = " ".join(list(seq_1l))
    encoded = tokenizer(seq_spaced, return_tensors="pt", add_special_tokens=True)

    input_ids = encoded["input_ids"][0]
    attention_mask = encoded["attention_mask"][0]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    print("Original sequence:", seq_1l)
    print("Residue length:", len(seq_1l))
    print("Total tokens:", len(tokens))
    print("Masked tokens:", int(attention_mask.sum().item()))

    print("\nTokens:")
    for i, tok in enumerate(tokens):
        print(i, tok)

    residue_tokens = [t.replace("▁", "") for t in tokens if t not in tokenizer.all_special_tokens]
    print("\nReconstructed:", "".join(residue_tokens))
    print("Match:", "".join(residue_tokens) == seq_1l)

inspect_prott5_tokenization(seq1)

In [ ]:
# ProtT5 residue embeddings
def prott5_residue_embeddings(seq_1l: str):
    seq_spaced = " ".join(list(seq_1l))
    encoded = tokenizer(seq_spaced, return_tensors="pt", add_special_tokens=True)

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    hidden = outputs.last_hidden_state[0]
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    keep_idx = [i for i, tok in enumerate(tokens) if tok not in tokenizer.all_special_tokens]
    residue_emb = hidden[keep_idx].detach().cpu().numpy()

    if residue_emb.shape[0] != len(seq_1l):
        raise ValueError(f"Embeddings {residue_emb.shape[0]} != residues {len(seq_1l)}")

    return residue_emb

emb = prott5_residue_embeddings(seq1)
print("Embedding shape:", emb.shape)

np.save(f"{OUTPUT_PREFIX}_prott5_embeddings.npy", emb)

residue_map = pd.DataFrame({
    "residue_index_1based": np.arange(1, len(seq1) + 1),
    "residue_index_0based": np.arange(len(seq1)),
    "residue_1l": list(seq1),
})
residue_map.to_csv(f"{OUTPUT_PREFIX}_residue_map.csv", index=False)

In [ ]:
# Save metadata
meta = {
    **selection_info,
    "pocket_radius": POCKET_RADIUS,
    "n_residues": len(seq1),
    "pocket_seq_1L": seq1,
    "pocket_seq_3L": " ".join(seq3),
    "embeddings_file": f"{OUTPUT_PREFIX}_prott5_embeddings.npy",
}
pd.DataFrame([meta]).to_csv(f"{OUTPUT_PREFIX}_metadata.csv", index=False)

print("Saved files:")
print("-", f"{OUTPUT_PREFIX}_residues.csv")
print("-", f"{OUTPUT_PREFIX}_sequence_1l.txt")
print("-", f"{OUTPUT_PREFIX}_sequence_3l.txt")
print("-", f"{OUTPUT_PREFIX}_prott5_embeddings.npy")
print("-", f"{OUTPUT_PREFIX}_residue_map.csv")
print("-", f"{OUTPUT_PREFIX}_metadata.csv")